<a href="https://colab.research.google.com/github/deepanshuMeteor/AI-For-SoftwareDeveloper/blob/main/jira_agent_test_generation_lab_(1).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Lab: Automate Test Case Generation from Jira Stories using Open‑Source LLMs

In this lab you will build a small **agent** that:

1. Connects to a Jira project and fetches a user story by key.  
2. Extracts requirement‑style information from the story (summary, description, acceptance criteria).  
3. Uses an **open‑source LLM** (from Hugging Face) to generate structured test cases.  

> ⚠️ **Note:** This notebook is meant as a starting point. You will still need valid Jira credentials and a Jira Cloud project.


## 1. Architecture Overview

High‑level flow:

1. **Config & Secrets** – Configure Jira URL, user/email, and API token (kept outside the code when possible).  
2. **Jira Connector (Tool)** – A Python function `fetch_jira_issue(issue_key)` that calls the Jira REST API.  
3. **Requirement Extractor** – Function that pulls `summary`, `description`, and (optionally) acceptance criteria from the Jira issue JSON.  
4. **LLM Wrapper** – An open‑source LLM loaded via `transformers` (e.g., `TinyLlama/TinyLlama-1.1B-Chat-v1.0`).  
5. **Test Generation Agent** – Orchestrator that:
   - fetches the story →
   - extracts requirements →
   - builds a prompt →
   - calls the LLM →
   - returns test cases in a Markdown table.


## Story details

**Summary**  
Enable Dark/Light Mode switching

**Issue Type**  
Story  

**Description**
I want to enable switching between dark and light modes in website

## 2. Environment Setup

Run the cell below **once** to install dependencies (uncomment if needed).  
If you're using a managed environment that already has `transformers` and `torch`, you can skip the install.


In [ ]:
# If you're in a fresh environment (e.g., Colab), uncomment and run:
!pip install -q transformers torch requests


## 3. Imports


In [ ]:
import os
import json
import getpass
from typing import Dict, Any

import requests
import torch
from transformers import pipeline


## 4. Jira Configuration

We will read Jira configuration from environment variables to avoid hard‑coding secrets.

Required values:

- `JIRA_BASE_URL` – e.g. `https://your-domain.atlassian.net`
- `JIRA_EMAIL` – your Jira login email / username
- `JIRA_API_TOKEN` – Jira API token (you can generate one from your Atlassian account)

You can either:

- Set them in your shell _before_ starting Jupyter, or  
- Input them interactively in this notebook (not saved to file).

### Access API Token and create a story
#### API Token
- Go to https://id.atlassian.com/manage-profile/security/api-tokens -> Create API Token -> Create -> Copy the API token

#### Create a story
- With an existing Jira account log in to https://your-domain.atlassian.net (e.g. https://meteoros-team-jhc5bzmb.atlassian.net/)

- Click on Create adjacent to search bar -> Type "Story" -> Enter Summary and Description and save

- Note down the story key (e.g. KAN-4)



In [ ]:
# --- Jira configuration ---

os.environ["JIRA_BASE_URL"] = "https://aiuser1.atlassian.net/"  # TODO: replace
os.environ["JIRA_EMAIL"] = "anshu0105@gmail.com"                # TODO: replace
os.environ["JIRA_API_TOKEN"] = ""

# Ask for API token if it's not already set as an env var
if not os.environ.get("JIRA_API_TOKEN"):
    print("JIRA_API_TOKEN is not set in environment. It will be requested interactively.")
    os.environ["JIRA_API_TOKEN"] = getpass.getpass("Enter your Jira API token (input hidden): ")

JIRA_API_TOKEN = os.environ["JIRA_API_TOKEN"]



## 5. Jira Connector "Tool"

A minimal helper that calls the Jira REST API and returns the issue JSON.  
We are using the Jira Cloud v3 API: `/rest/api/3/issue/{issueKey}`.


In [ ]:
def fetch_jira_issue(issue_key: str) -> Dict[str, Any]:
    """
    Fetch a Jira issue by key (e.g. "PROJ-123") and return the JSON response.
    """
    url = f"{os.environ.get("JIRA_BASE_URL")}rest/api/3/issue/{issue_key}"
    auth = (os.environ.get("JIRA_EMAIL"), JIRA_API_TOKEN)

    response = requests.get(url, auth=auth)
    try:
        response.raise_for_status()
    except requests.HTTPError as e:
        print("Error while calling Jira:", e)
        print("Response text:", response.text[:1000])
        raise

    return response.json()


# Quick smoke test (update ISSUE_KEY to a real story in your project)
TEST_ISSUE_KEY = "KAN-1"  # TODO: replace with a real Jira key, e.g. "KAN-4"
try:
    test_issue = fetch_jira_issue(TEST_ISSUE_KEY)
    print(f"Fetched issue {TEST_ISSUE_KEY}. Issue type:", test_issue.get("fields", {}).get("issuetype", {}).get("name"))
except Exception as e:
    print("Test fetch failed (this is expected until you provide a real issue key).")
    print("Error:", e)


Fetched issue KAN-1. Issue type: Task


## 6. Requirement Extractor

Jira fields are flexible and can include rich‑text content. In this lab we will:

- Always extract `summary`
- Convert `description` to a string
- Optionally extract **acceptance criteria** if your project uses a custom field for it

> 🔧 You may need to adjust the `AC_FIELD_ID` value below to match your Jira instance (e.g. `customfield_10026`).


In [ ]:
def normalize_field(field_value) -> str:
    """
    Convert a Jira field (which may be rich text / nested JSON) into a plain string.
    For simplicity, we just JSON-dump non-string fields.
    """
    if field_value is None:
        return ""
    if isinstance(field_value, str):
        return field_value
    # For rich-text / structured fields, a simple JSON dump is fine for the prompt.
    return json.dumps(field_value, indent=2)


def extract_requirements(issue_json: Dict[str, Any]) -> Dict[str, str]:
    fields = issue_json.get("fields", {})

    summary = fields.get("summary", "")
    description_raw = fields.get("description")


    return {
        "summary": normalize_field(summary),
        "description": normalize_field(description_raw),
    }


# Example (will only work after TEST_ISSUE_KEY is set to a real issue)
if 'test_issue' in globals():
    reqs_preview = extract_requirements(test_issue)
    print("Summary:\n", reqs_preview["summary"], "\n")
    print("Description (truncated):\n", reqs_preview["description"][:500], "\n")


Summary:
 Enable Dark/Light Mode switching 

Description (truncated):
 {
  "type": "doc",
  "version": 1,
  "content": [
    {
      "type": "paragraph",
      "content": [
        {
          "type": "text",
          "text": "I want to enable switching between dark and light modes in website"
        }
      ],
      "attrs": {
        "localId": "d33c987acc59"
      }
    }
  ]
} 



## 7. Open‑Source LLM Setup

We will use a small, openly available instruction‑tuned model from Hugging Face via `transformers.pipeline`.

You can swap `MODEL_NAME` for any other open‑source model you have resources for (e.g., `mistralai/Mistral-7B-Instruct-v0.2` if you have a GPU).


In [ ]:
MODEL_NAME = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"  # Small, open model; feel free to change

device = 0 if torch.cuda.is_available() else -1
print("Using device:", "cuda:0" if device == 0 else "cpu")

llm = pipeline(
    "text-generation",
    model=MODEL_NAME,
    tokenizer=MODEL_NAME,
    device=device,
)


def generate_with_llm(prompt: str, max_new_tokens: int = 512) -> str:
    """
    Simple wrapper around the HF pipeline for text generation.
    Returns only the newly generated part (prompt stripped from the output).
    """
    outputs = llm(
        prompt,
        max_new_tokens=max_new_tokens,
        do_sample=True,
        temperature=0.2,
        top_p=0.9,
        return_full_text=False
    )
    full_text = outputs[0]["generated_text"]
    # Strip the prompt from the front if the model echoes it
    if full_text.startswith(prompt):
        return full_text[len(prompt):].strip()
    return full_text.strip()




Using device: cuda:0


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/608 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.20G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/551 [00:00<?, ?B/s]

## 8. Prompt Engineering for Test Generation

We will instruct the model to act as a **senior QA engineer** and to output test cases in a Markdown table.

Columns:

- `ID`
- `Title`
- `Type (Positive/Negative/Edge)`
- `Pre-conditions`
- `Steps`
- `Expected Result`
- `Priority`


In [ ]:
BASE_TESTGEN_PROMPT = """
You are a senior QA engineer.
Your task:
- Read the Jira story requirements below.
- Generate a set of 3 test cases that validate the story.
- Cover positive, negative, and edge/boundary scenarios.
- Use concise, professional language.
Format your answer as a Markdown table with the columns:
| ID | Title | Type (Positive/Negative/Edge) | Steps | Expected Result | Priority |
Jira story requirements:
Summary:
{summary}
Description:
{description}
Now produce the Markdown table of test cases.
"""


def build_testcase_prompt(requirements: Dict[str, str]) -> str:
    return BASE_TESTGEN_PROMPT.format(**requirements)


## 9. Putting It Together: The Test Generation Agent

The "agent" here is a simple orchestrator that:

1. Fetches the Jira story.  
2. Extracts requirement fields.  
3. Builds the LLM prompt.  
4. Calls the model and returns the generated test cases.


In [ ]:
class JiraTestGenerationAgent:
    def __init__(self, issue_key: str):
        self.issue_key = issue_key

    def run(self) -> str:
        # Step 1: Fetch Jira story
        issue_json = fetch_jira_issue(self.issue_key)

        # Step 2: Extract requirements
        requirements = extract_requirements(issue_json)

        # Step 3: Build prompt
        prompt = build_testcase_prompt(requirements)

        print("====== Prompt sent to LLM (truncated) ======")
        print(prompt[:1000], "...\n")

        # Step 4: Call LLM
        print("====== Generated Test Cases ======\n")
        output = generate_with_llm(prompt)
        print(output)
        return output


## 10. Run the Agent on a Real Jira Story

Set `ISSUE_KEY` to a real Jira issue (e.g., `KAN-4`, `PROJ-101`, etc.), then run the cell.


In [ ]:
import gc, torch
gc.collect()
torch.cuda.empty_cache()

In [ ]:
ISSUE_KEY = "KAN-1"
agent = JiraTestGenerationAgent(issue_key=ISSUE_KEY)
testcases_markdown = agent.run()


Passing `generation_config` together with generation-related arguments=({'do_sample', 'max_new_tokens', 'temperature', 'top_p'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
Both `max_new_tokens` (=512) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


====== Prompt sent to LLM (truncated) ======

You are a senior QA engineer.
Your task:
- Read the Jira story requirements below.
- Generate a set of 3 test cases that validate the story.
- Cover positive, negative, and edge/boundary scenarios.
- Use concise, professional language.
Format your answer as a Markdown table with the columns:
| ID | Title | Type (Positive/Negative/Edge) | Steps | Expected Result | Priority |
Jira story requirements:
Summary:
Enable Dark/Light Mode switching
Description:
{
  "type": "doc",
  "version": 1,
  "content": [
    {
      "type": "paragraph",
      "content": [
        {
          "type": "text",
          "text": "I want to enable switching between dark and light modes in website"
        }
      ],
      "attrs": {
        "localId": "d33c987acc59"
      }
    }
  ]
}
Now produce the Markdown table of test cases.
 ...

====== Generated Test Cases ======

| ID | Title | Type (Positive/Negative/Edge) | Steps | Expected Result | Priority |
|----|----

In [ ]:
import csv
import json

# ---------- 1. Parse Markdown table into rows ----------

lines = [line.strip() for line in testcases_markdown.splitlines()]

# Keep only the lines that look like table rows
table_lines = [
    line for line in lines
    if "|" in line and not line.lstrip().startswith("|---")
]

if not table_lines:
    raise ValueError("No markdown table lines found in testcases_markdown")

# First row = header
header_cells = [h.strip() for h in table_lines[0].split("|") if h.strip()]
headers = header_cells

rows = []
for row_line in table_lines[1:]:
    # Skip separator or empty-ish lines
    if set(row_line.replace("|", "").replace("-", "").strip()) == set():
        continue

    cells = [c.strip() for c in row_line.split("|") if c.strip()]
    # Pad / trim to match header length
    if len(cells) < len(headers):
        cells += [""] * (len(headers) - len(cells))
    elif len(cells) > len(headers):
        cells = cells[: len(headers)]

    row_dict = dict(zip(headers, cells))
    rows.append(row_dict)

print(f"Parsed {len(rows)} test cases with columns: {headers}")

# ---------- 2. Save as CSV ----------

csv_filename = f"{ISSUE_KEY}_testcases.csv"

with open(csv_filename, "w", newline="", encoding="utf-8") as csvfile:
    writer = csv.DictWriter(csvfile, fieldnames=headers)
    writer.writeheader()
    writer.writerows(rows)

print(f"✅ CSV saved to {csv_filename}")

# ---------- 3. Save as JSON ----------

json_filename = f"{ISSUE_KEY}_testcases.json"

with open(json_filename, "w", encoding="utf-8") as jsonfile:
    json.dump(rows, jsonfile, ensure_ascii=False, indent=2)

print(f"✅ JSON saved to {json_filename}")


Parsed 11 test cases with columns: ['ID', 'Title', 'Type (Positive/Negative/Edge)', 'Steps', 'Expected Result', 'Priority']
✅ CSV saved to KAN-1_testcases.csv
✅ JSON saved to KAN-1_testcases.json


## 11. Extensions & Next Steps

Ideas for extending this lab:

- **Code‑Change Driven Tests**  
  Instead of (or in addition to) Jira stories, parse Git diffs and feed changed functions/classes to the LLM to suggest tests.

- **Save Results Back to Jira**  
  Use the Jira REST API to post the generated test cases as a comment or attach them as a file to the story.

- **Stronger Parsing**  
  Parse Jira rich‑text descriptions and structured acceptance criteria fields more carefully instead of simple JSON dumps.

- **Multi‑Agent Setup**  
  Split responsibilities into multiple agents:
  - _Requirements Agent_ – cleans & normalizes story text.
  - _Test Designer Agent_ – proposes tests.
  - _Reviewer Agent_ – checks coverage and suggests missing cases.
